In [ ]:
import io
import os
import time
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
import requests

from utils_firerisk import REGION, canon_key, load_oni, local_dt, random_points_in, sample_tile, tile_name

DATA = Path("../data")
RAW, INTERIM = DATA / "raw", DATA / "interim"
for folder in (RAW / "firms", RAW / "wx", INTERIM):
    folder.mkdir(parents=True, exist_ok=True)

START_DATE, END_DATE = "2012-01-20", "2025-12-31"
PRODUCT = "VIIRS_SNPP_SP"
TREE_FRAC_MIN = 0.5
N_RANDOM_PTS = 1500
RUN_SUBSET = None
FIRMS_MAP_KEY = os.environ.get("FIRMS_MAP_KEY", "")

In [ ]:
def load_boundaries():
    meta = requests.get("https://www.geoboundaries.org/api/current/gbOpen/VNM/ADM1/", timeout=60).json()
    return gpd.read_file(meta["gjDownloadURL"])


adm1 = load_boundaries().to_crs(4326)
name_col = "shapeName" if "shapeName" in adm1.columns else adm1.columns[0]
adm1["key"] = adm1[name_col].map(canon_key).replace({"con dao": "ba ria vung tau"})
adm1 = adm1.dissolve(by="key", as_index=False)
print("Số đơn vị ADM1:", len(adm1))
print("Có trong ranh giới nhưng không có trong bảng vùng:", sorted(set(adm1["key"]) - set(REGION)))
print("Có trong bảng vùng nhưng không khớp ranh giới:", sorted(set(REGION) - set(adm1["key"])))

adm1 = adm1[adm1["key"].isin(REGION)].copy()
if RUN_SUBSET:
    adm1 = adm1[adm1["key"].isin(RUN_SUBSET)].copy()

centers = adm1.geometry.representative_point()
PROV = pd.DataFrame({
    "key": adm1["key"].values,
    "region": adm1["key"].map(REGION).values,
    "rep_lat": centers.y.values,
    "rep_lon": centers.x.values,
    "area_km2": (adm1.to_crs(32648).geometry.area / 1e6).values,
}).reset_index(drop=True)
polygons = dict(zip(adm1["key"], adm1.geometry))

minx, miny, maxx, maxy = adm1.total_bounds
BBOX = f"{minx - 0.1:.3f},{miny - 0.1:.3f},{maxx + 0.1:.3f},{maxy + 0.1:.3f}"
print(len(PROV), "tỉnh | BBOX:", BBOX)

Ranh giới khớp đủ 63 tỉnh với bảng vùng, không tỉnh nào lệch tên. Bản geoBoundaries đã sáp nhập (khoảng 34 đơn vị) sẽ không dùng được.

In [ ]:
def firms_chunk(start, days=5):
    path = RAW / "firms" / f"{PRODUCT}_{BBOX.replace(',', '_')}_{start}_{days}.csv"
    if path.exists() and path.stat().st_size > 0:
        return pd.read_csv(path)

    url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{FIRMS_MAP_KEY}/{PRODUCT}/{BBOX}/{days}/{start}"
    for attempt in range(6):
        try:
            response = requests.get(url, timeout=180)
            if response.status_code == 200 and response.text.startswith("latitude"):
                path.write_text(response.text, encoding="utf-8")
                return pd.read_csv(io.StringIO(response.text))
            print("phản hồi lạ:", response.status_code, response.text[:120].replace("\n", " "))
        except requests.RequestException as error:
            print("lỗi mạng:", error)
        time.sleep(15 * (attempt + 1))
    raise RuntimeError(f"FIRMS thất bại tại {start}")


def load_firms():
    starts = pd.date_range(START_DATE, END_DATE, freq="5D")
    chunks = []
    for i, start in enumerate(starts):
        chunks.append(firms_chunk(start.strftime("%Y-%m-%d")))
        if i % 100 == 0:
            print(i, "/", len(starts), start.date())
    hot = pd.concat(chunks, ignore_index=True)
    hot = hot.drop_duplicates(["latitude", "longitude", "acq_date", "acq_time", "satellite"])
    return hot[hot["acq_date"] <= END_DATE].copy()

FIRMS chỉ nhận tối đa 5 ngày mỗi lần gọi nên 14 năm là 1.019 khối. Một số khối gặp lỗi tạm thời của máy chủ (504, Invalid MAP_KEY) và được thử lại thành công. Tổng cộng 5.455.690 điểm cháy thô từ 2012-01-20 đến 2025-12-31, phần lớn có độ tin cậy nominal (4,72 triệu), low 0,50 triệu, high 0,24 triệu.

In [ ]:
WORLDCOVER_URL = "https://esa-worldcover.s3.eu-central-1.amazonaws.com/v200/2021/map/ESA_WorldCover_10m_2021_v200_{tile}_Map.tif"
HOTSPOTS_FILE = INTERIM / "hotspots_enriched.csv"
CONFIDENCE_OK = ["n", "h", "nominal", "high"]


def sample_worldcover(lats, lons):
    lats, lons = np.asarray(lats), np.asarray(lons)
    out = np.zeros(len(lats), dtype="uint8")
    tile_lat = (np.floor(lats / 3) * 3).astype(int)
    tile_lon = (np.floor(lons / 3) * 3).astype(int)
    env = rasterio.Env(
        GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR",
        CPL_VSIL_CURL_ALLOWED_EXTENSIONS=".tif",
        GDAL_HTTP_MAX_RETRY="5",
        GDAL_HTTP_RETRY_DELAY="3",
    )
    with env:
        for la, lo in set(zip(tile_lat.tolist(), tile_lon.tolist())):
            rows = np.where((tile_lat == la) & (tile_lon == lo))[0]
            with rasterio.open(WORLDCOVER_URL.format(tile=tile_name(la, lo))) as src:
                out[rows] = sample_tile(src, lons[rows], lats[rows])
    return out


def add_flags(pts):
    pts["forest"] = pts["tree_frac"] >= TREE_FRAC_MIN
    pts["conf_ok"] = pts["confidence"].astype(str).str.lower().isin(CONFIDENCE_OK)
    return pts


def load_cached_hotspots():
    if not HOTSPOTS_FILE.exists():
        return None
    cached = pd.read_csv(HOTSPOTS_FILE, parse_dates=["ltime", "ldate"])
    has_provinces = set(PROV["key"]) <= set(cached["prov"])
    has_columns = {"tree_frac", "lc_center", "confidence"} <= set(cached.columns)
    if not (has_provinces and has_columns):
        return None
    return add_flags(cached[cached["prov"].isin(PROV["key"])].copy())


def tree_cover_around(points, radius_m=190.0, chunk=400_000):
    angles = np.arange(8) * np.pi / 4
    offsets = [(0.0, 0.0)] + [(radius_m * np.sin(a), radius_m * np.cos(a)) for a in angles]
    tree_frac, center_class = [], []
    for start in range(0, len(points), chunk):
        part = points.iloc[start:start + chunk]
        lat0, lon0 = part["latitude"].values, part["longitude"].values
        lats = np.concatenate([lat0 + dn / 110570.0 for dn, de in offsets])
        lons = np.concatenate([lon0 + de / (111320.0 * np.cos(np.radians(lat0))) for dn, de in offsets])
        classes = sample_worldcover(lats, lons).reshape(len(offsets), len(part)).T
        tree_frac.append((classes == 10).mean(axis=1))
        center_class.append(classes[:, 0])
        print("WorldCover", start + len(part), "/", len(points))
    return np.concatenate(tree_frac), np.concatenate(center_class)


def enrich_hotspots(raw_hot):
    pts = gpd.GeoDataFrame(raw_hot, geometry=gpd.points_from_xy(raw_hot["longitude"], raw_hot["latitude"]), crs=4326)
    pts = gpd.sjoin(pts, adm1[["key", "geometry"]], predicate="within", how="inner")
    pts = pts.drop(columns=["index_right", "geometry"]).rename(columns={"key": "prov"})
    pts["ltime"] = local_dt(pts)
    pts["ldate"] = pts["ltime"].dt.normalize()
    pts["lhour"] = pts["ltime"].dt.hour
    print("Điểm trong các tỉnh chọn:", len(pts))

    unique = pts[["latitude", "longitude"]].round(4).drop_duplicates().reset_index(drop=True)
    unique["tree_frac"], unique["lc_center"] = tree_cover_around(unique)

    pts["latr"] = pts["latitude"].round(4)
    pts["lonr"] = pts["longitude"].round(4)
    pts = pts.merge(unique.rename(columns={"latitude": "latr", "longitude": "lonr"}), on=["latr", "lonr"], how="left")
    return add_flags(pts)


hotspots = load_cached_hotspots()
if hotspots is None:
    hotspots = enrich_hotspots(load_firms())
    hotspots.to_csv(HOTSPOTS_FILE, index=False)
print(hotspots.groupby("prov")[["forest", "conf_ok"]].mean().round(3).head(10))

947.287 điểm nằm trong 63 tỉnh (943.107 tọa độ duy nhất đưa vào WorldCover, mất khoảng 24 phút). Tỉ lệ điểm cháy rơi vào rừng khác nhau rất rõ giữa các tỉnh: An Giang 0,9%, Bạc Liêu 0,8%, Bắc Kạn 97,9%.

In [ ]:
STATIC_FILE = INTERIM / "static_prov.csv"
SHARE_COLUMNS = ["forest_share", "crop_share", "grass_share", "built_share"]


def land_cover_shares():
    rows = []
    for key in PROV["key"]:
        xy = random_points_in(polygons[key], N_RANDOM_PTS, seed=7)
        classes = sample_worldcover(xy[:, 1], xy[:, 0])
        rows.append({
            "key": key,
            "forest_share": (classes == 10).mean(),
            "crop_share": (classes == 40).mean(),
            "grass_share": (classes == 30).mean(),
            "built_share": (classes == 50).mean(),
        })
    return pd.DataFrame(rows)


cached = pd.read_csv(STATIC_FILE) if STATIC_FILE.exists() else None
if cached is not None and set(PROV["key"]) <= set(cached["key"]):
    shares = cached[["key"] + SHARE_COLUMNS]
else:
    shares = land_cover_shares()
static = PROV.merge(shares, on="key")
static["rep_elev_m"] = np.nan
print(static.round(3).head(10).to_string())

In [ ]:
WX_VARS = ["temperature_2m", "dew_point_2m", "precipitation"]
WX_URL = "https://archive-api.open-meteo.com/v1/archive"


def wx_year(name, lat, lon, year):
    path = RAW / "wx" / f"{name}_{year}.csv"
    if path.exists() and path.stat().st_size > 0:
        return pd.read_csv(path)

    params = dict(
        latitude=round(lat, 3), longitude=round(lon, 3),
        start_date=f"{year}-01-01", end_date=f"{year}-12-31",
        hourly=",".join(WX_VARS), models="era5", timezone="Asia/Bangkok",
    )
    for attempt in range(8):
        response = requests.get(WX_URL, params=params, timeout=180)
        is_json = response.headers.get("content-type", "").startswith("application/json")
        body = response.json() if is_json else {}
        if response.status_code == 200 and "hourly" in body:
            hourly = body["hourly"]
            frame = pd.DataFrame({"time": hourly["time"]})
            for var in WX_VARS:
                frame[var] = hourly[var if var in hourly else next(k for k in hourly if k.startswith(var))]
            frame["elevation"] = body.get("elevation")
            frame.to_csv(path, index=False)
            time.sleep(0.5)
            return frame

        reason = str(body.get("reason", ""))
        print("Open-Meteo:", response.status_code, reason[:120])
        if "Daily" in reason:
            raise RuntimeError("Hết hạn mức ngày của Open-Meteo, chạy lại sau 0h UTC; các tệp đã tải được giữ nguyên.")
        time.sleep(3700 if "Hourly" in reason else 65 if "Minutely" in reason else 30 * (attempt + 1))
    raise RuntimeError(f"Open-Meteo thất bại {name} {year}")


first_year = pd.Timestamp(START_DATE).year - 1
last_year = pd.Timestamp(END_DATE).year
started = time.time()
for _, prov in PROV.iterrows():
    for year in range(first_year, last_year + 1):
        frame = wx_year(prov["key"], prov["rep_lat"], prov["rep_lon"], year)
        if year == first_year:
            static.loc[static["key"] == prov["key"], "rep_elev_m"] = frame["elevation"].iloc[0]
    print(prov["key"], "| %.0fs" % (time.time() - started))

Open-Meteo giới hạn tần suất theo phút, giờ và ngày. Khi chạm hạn mức giờ hoặc ngày, chạy lại ô trên: các tệp đã tải trong data/raw/wx được dùng lại.

In [ ]:
PROV.to_csv(INTERIM / "prov_table.csv", index=False)
static.to_csv(STATIC_FILE, index=False)
load_oni(INTERIM / "oni.csv")